## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [57]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [58]:
pip install langchain_openai

In [59]:
pip install langchain_chroma


In [60]:
pip install langchain_huggingface

In [61]:
# We are using a local, low-cost model through Ollama

MODEL = "llama3.2"
db_name = "vector_db"

from openai import OpenAI

ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

print(f"Using Ollama model: {MODEL}")

Using Ollama model: llama3.2


In [62]:
# How many characters in all the documents?

import glob
import os

knowledge_base_path = "/knowledge-base/**/*"
files = glob.glob(knowledge_base_path, recursive=True)

# Keep only actual files, not directories
files = [
    file_path
    for file_path in files
    if os.path.isfile(file_path)
]

print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(
    f"Total characters in knowledge base: "
    f"{len(entire_knowledge_base):,}"
)

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [63]:
import zipfile

with zipfile.ZipFile("/content/knowledge-base.zip", "r") as zip_ref:
    zip_ref.extractall("/")

In [64]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [65]:
pip install langchain_community

In [66]:
from langchain_core.documents import Document

documents = []

for file_path in files:
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    documents.append(
        Document(
            page_content=content,
            metadata={"source": file_path}
        )
    )

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [67]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 413 chunks
First chunk:

page_content='# Careers at Insurellm

## Why Join Insurellm?

At Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.

After reaching 200 employees in 2020, we strategically restructured in 2022-2023 to focus on sustainable growth, operational excellence, and building a world-class remote-first culture. Today, we're a tight-knit team of exceptional professionals who deliver outsized impact through automation, AI, and strategic focus on high-value enterprise clients—from regional insurers to global reinsurance partners.

### Our Culture' metadata={'source': '/knowledge-base/company/careers.md'}


In [68]:
import os

db_name = "/content/vector_db"

# Pick an embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# Delete existing vector database if it exists
if os.path.exists(db_name):
    Chroma(
        persist_directory=db_name,
        embedding_function=embeddings
    ).delete_collection()

# Create the vectorstore
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=db_name
)

print(
    f"Vectorstore created with "
    f"{vectorstore._collection.count()} documents"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vectorstore created with 413 documents


In [69]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 384 dimensions in the vector store


In [76]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [71]:
import subprocess
import time

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

In [72]:
!ollama pull llama3.2

In [73]:
!ollama list

NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [78]:
!ollama run llama3.2 "Who is Avery?"

There are several individuals named Avery, so it's possible that you're ref
referring to one of the following:

1. Avery Williams: An American football wide receiver who plays for the Min
Minnesota Vikings.
2. Chris Avery: A British businessman and the CEO of Thomas Cook Group Hold
Holdings.
3. Jason Avery: An American football linebacker who played in the National 
Football League (NFL).
4. Katie Avery: An American actress, voice actress, and singer.
5. Ryan Avery: A Canadian politician who serves as a member of the Legislat
Legislative Assembly of Newfoundland and Labrador.

If you could provide more context or information about who Avery is that yo
you're referring to, I'd be happy to try and help you further.



In [77]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

response = llm.invoke("Who is Avery?")
print(response.content)

There are several notable individuals named Avery, so it's possible that you're referring to one of the following:

1. Avery Brooks: An American actor known for his roles in TV shows such as "Star Trek: Deep Space Nine" and "Designated Survivor".
2. Avery Whitted: An American professional poker player who has won several tournaments, including the World Series of Poker (WSOP) Main Event.
3. Avery Wines: A wine company that produces a range of wines under various labels, including Avery, which is their flagship brand.
4. Avery Dennison: A global materials science company that develops and manufactures adhesives, films, and other specialty materials.

If you could provide more context or information about the Avery you're referring to, I'd be happy to try and provide a more specific answer.


In [79]:
!pip install -U langchain-ollama

In [80]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 137 not upgraded.


In [81]:
pip install langchain_huggingface

In [82]:
pip install langchain_chroma

In [83]:
pip install langchain_openai

In [84]:
!pip install -U langchain-ollama

In [85]:
MODEL = "llama3.2"
DB_NAME = "vector_db"

load_dotenv(override=True)

False

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [86]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [87]:
from langchain_ollama import ChatOllama

retriever = vectorstore.as_retriever()

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

### These LangChain objects implement the method `invoke()`

In [89]:
retriever.invoke("Who is Avery?")

[Document(id='18e9b327-26d3-4a9c-b95d-f5083d7bb6b0', metadata={'source': '/knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and adaptability throughout her

In [90]:
llm.invoke("Who is Avery Lancaster?")

AIMessage(content="I couldn't find any information on a person named Avery Lancaster. It's possible that Avery Lancaster is a private individual or not a public figure, or they may not have a significant online presence.\n\nHowever, I did find information on Avery Lancaster, the American soccer player who plays as a midfielder for the United States women's national team.", additional_kwargs={}, response_metadata={'model': 'llama3.2', 'created_at': '2026-08-08T12:55:22.164247359Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13912228572, 'load_duration': 312241846, 'prompt_eval_count': 30, 'prompt_eval_duration': 1188995000, 'eval_count': 68, 'eval_duration': 12387595000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'}, id='lc_run--019fe171-04d9-7963-a0d4-293198ff670f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 68, 'total_tokens': 98})

## Time to put this together!

In [91]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [92]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [93]:
answer_question("Who is Averi Lancaster?", [])

"Avery Lancaster is actually Emily Carter's colleague and not a public figure I have information on. However, based on the context provided, it seems that Avery Lancaster is likely an employee or colleague of Emily Carter at Insurellm.\n\nIf you're looking for more information about Emily Carter, she is indeed a valuable asset to the company, known for her strategic thinking and ability to balance customer needs with business objectives."

## What could possibly come next? 😂

In [94]:
gr.ChatInterface(answer_question).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5219b4a9c56750ef72.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [95]:
docs = retriever.invoke("Who is Avery Lancaster?")

print("Number of documents:", len(docs))
print("Documents:", docs)

Number of documents: 4
Documents: [Document(id='18e9b327-26d3-4a9c-b95d-f5083d7bb6b0', metadata={'source': '/knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilien

In [96]:
import glob

files = glob.glob("/knowledge-base/*", recursive=True)

print("Number of files:", len(files))
print(files[:10])

Number of files: 4
['/knowledge-base/company', '/knowledge-base/contracts', '/knowledge-base/employees', '/knowledge-base/products']


In [98]:
from langchain_community.document_loaders import DirectoryLoader

documents = []

folders = glob.glob("knowledge/*")

for folder in folders:
    doc_type = os.path.basename(folder)

    loader = DirectoryLoader(
        folder,
        glob="**/*.md"
    )

    docs = loader.load()

    for doc in docs:
        doc.metadata["doc_type"] = doc_type

    documents.extend(docs)

print("Documents loaded:", len(documents))

Documents loaded: 0


## Admit it - you thought RAG would be more complicated than that!!